# 章节实践

## 概述

通过本章的学习，你已经掌握了pyasc核函数开发的完整流程：`@asc.jit`装饰器、Tensor与数据搬运、手动流水同步、核函数调用与验证。现在通过开发Mul算子来巩固所学知识。

### 实践任务：开发Mul算子

基于本章学习的手动同步流水方式，实现一个**Mul算子**（矢量乘法算子），算子命名为`vmul_kernel`。

### 算子描述

```text
z = x * y
```

计算逻辑与Add算子类似：从Global Memory搬运数据到Local Memory，使用矢量计算接口完成两个输入相乘，再将结果搬运到Global Memory。

### 算子规格

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">项目</th>
      <th align="left">取值</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>输入x</td><td>shape: (8, 2048), dtype: float32, format: ND</td></tr>
    <tr><td>输入y</td><td>shape: (8, 2048), dtype: float32, format: ND</td></tr>
    <tr><td>输出z</td><td>shape: (8, 2048), dtype: float32, format: ND</td></tr>
    <tr><td>核函数名</td><td><code>vmul_kernel</code></td></tr>
    <tr><td>核数</td><td>8</td></tr>
    <tr><td>缓冲区数</td><td>2（双缓冲）</td></tr>
    <tr><td>主要接口</td><td><code>asc.data_copy</code>, <code>asc.mul</code>, <code>asc.set_flag</code>/<code>wait_flag</code></td></tr>
  </tbody>
</table>

### 要求

- 使用**手动同步流水**方式（set_flag/wait_flag）实现
- 支持Float（float32）类型输入输出
- 使用torch.allclose验证结果正确性

### 提示

- 参考`src/add.py`样例代码
- 将`asc.add`替换为`asc.mul`（矢量乘法接口）
- 核函数名从`vadd_kernel`改为`vmul_kernel`

---

请在下方代码 cell 中编写你的Mul算子实现：

### 开发步骤

1. **编写核函数 `vmul_kernel`**：复制 `vadd_kernel` 的结构，将 `asc.add` 替换为 `asc.mul`
2. **编写 Launch 函数 `vmul_launch`**：复制 `vadd_launch`，调用 `vmul_kernel`
3. **编写验证函数 `vmul_custom`**：生成测试数据，调用 `vmul_launch`，使用 `torch.allclose(z, x * y)` 验证
4. **运行验证**：取消下方运行命令的注释，执行验证


In [ ]:
# 环境初始化
!mkdir -p Sources/02.07

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

In [ ]:
%%writefile Sources/02.07/mul_manual.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.

import logging
import argparse
import torch
try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt

USE_CORE_NUM = 8
BUFFER_NUM = 2
TILE_NUM = 8

logging.basicConfig(level=logging.INFO)


@asc.jit
def vmul_kernel(x: asc.GlobalAddress, y: asc.GlobalAddress, z: asc.GlobalAddress, block_length: int):
    # 请参考 add.py 中的 vadd_kernel 实现，将 asc.add 替换为 asc.mul
    pass


def vmul_launch(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # 请参考 add.py 中的 vadd_launch 实现，调用 vmul_kernel
    pass


def vmul_custom(backend: config.Backend, platform: config.Platform):
    # 请参考 add.py 中的 vadd_custom 实现，验证结果使用 torch.allclose(z, x * y)
    pass


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError("Unsupported Backend! Supported: ['Model', 'NPU']")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [platform.value for platform in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample mul.")
    vmul_custom(backend, platform)
    logging.info("[INFO] Sample mul run success.")

---

编写完成后，执行以下命令运行你的Mul算子：

In [ ]:
# 运行Mul算子
!cd Sources/02.07 && python3 mul_manual.py -r NPU

---

执行以下代码获取参考答案。

In [ ]:
!cat ./answer/02.07_chapter_practice/mul_manual.py